## Grid Evaluation
---

Visualize t_start x t_end ablation grids for each sample: load cell images from a grid
run folder, overlay `TARGET_METRIC` as a viridis tint, highlight reference and
best-scoring cells, and render composite figures with prompts from the PIE-Bench mapping.

*Please note that the data for this notebook is not stored in the respoitory*

In [ ]:
"""
Configure matplotlib, import plotting and image libraries, define the t_start / t_end
sweep values used by the grid ablation, and import the active target-metric settings
from settings.py.
"""

import os
import sys

# Use a writable matplotlib cache dir (silences the default-path warning).
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from PIL import Image, ImageDraw

# t_start / t_end sweep values (matches scripts/run_grid_ablation.GRID_VALUES).
GRID_VALUES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from settings import TARGET_METRIC_COL, TARGET_METRIC_COL_FN, TARGET_METRIC_COL_LABEL, TARGET_T_DELTA


In [ ]:
def load_cell(cells_dir: Path, t_start: float, t_end: float):
    """Load a cell image from the given directory."""
    def _param_slug(param_name: str, value: float) -> str:
        return f"{param_name}_{value:.1f}".replace(".", "p")
    path = cells_dir / f"{_param_slug('t_start', t_start)}__{_param_slug('t_end', t_end)}.png"
    if not path.exists():
        return None
    with Image.open(path) as img:
        return img.convert("RGB")


def greyscale_image(img: Image.Image) -> Image.Image:
    return img.convert("L").convert("RGB")


def viridis_overlays(scores: dict, alpha: float = 0.45) -> dict:
    """Map (t_start, t_end) -> RGBA tint from normalized scores."""
    if not scores:
        return {}
    vals = list(scores.values())
    vmin, vmax = min(vals), max(vals)
    denom = vmax - vmin if vmax > vmin else 1.0
    cmap = plt.colormaps["viridis"]
    overlays = {}
    for key, val in scores.items():
        t = (val - vmin) / denom
        r, g, b, _ = cmap(t)
        overlays[key] = (int(r * 255), int(g * 255), int(b * 255), int(255 * alpha))
    return overlays


def grid_scores(df: pd.DataFrame, sample_id: str, t_delta: float, metric_col: str) -> dict:
    """Per-cell metric scores for one sample and t_delta slice."""
    sub = df[(df["sample_id"] == int(sample_id)) & (df["t_delta"] == float(t_delta))]
    return {
        (round(float(row["t_start"]), 1), round(float(row["t_end"]), 1)): float(row[metric_col])
        for row in sub.to_dict("records")
    }


def best_highlight(
    scores: dict,
    color: str = "#ff7f0e",
    exclude: tuple[float, float] | None = None,
) -> dict:
    """Highlight border for the highest-scoring cell, optionally skipping one key."""
    if not scores:
        return {}
    candidates = scores if exclude is None else {k: v for k, v in scores.items() if k != exclude}
    if not candidates:
        return {}
    return {max(candidates, key=candidates.get): color}


def build_grid_image(cells_dir: Path, values, thumb_px: int = 96, gap: int = 3,
                     highlights=None, color_border_px: int = 10, inner_border_px: int = 4,
                     t_delta: float = 0.0, scores=None, overlay_alpha: float = 0.45,
                     greyscale_keys: set | None = None):
    """Paste all cells into one composite (t_start = x, t_end up the y-axis).

    ``highlights`` maps (t_start, t_end) -> border color; matching cells get a
    colored frame drawn on their edge.

    ``scores`` maps (t_start, t_end) -> float; matching cells get a viridis tint.

    ``greyscale_keys`` lists (t_start, t_end) cells to render in greyscale
    without a viridis score overlay.
    """
    highlights = highlights or {}
    greyscale_keys = greyscale_keys or set()
    overlays = viridis_overlays(scores or {}, alpha=overlay_alpha)
    t_delta = float(t_delta)
    n = len(values)
    stride = thumb_px + gap
    size = n * stride + gap  # gap also frames the outer edge
    canvas = Image.new("RGB", (size, size), "#ffffff")
    draw = ImageDraw.Draw(canvas)
    found = False
    for row, t_end in enumerate(reversed(values)):  # top row = largest t_end
        for col, t_start in enumerate(values):
            x = gap + col * stride
            y = gap + row * stride
            key = (round(t_start, 1), round(t_end, 1))
            cell = load_cell(cells_dir, t_start, t_end)
            if cell is not None:
                found = True
                thumb = cell.resize((thumb_px, thumb_px), Image.Resampling.LANCZOS)
                if key in greyscale_keys:
                    thumb = greyscale_image(thumb)
                canvas.paste(thumb, (x, y))
                rgba = overlays.get(key)
                if rgba is not None and key not in greyscale_keys:
                    region = canvas.crop((x, y, x + thumb_px, y + thumb_px)).convert("RGBA")
                    tint = Image.new("RGBA", (thumb_px, thumb_px), rgba)
                    canvas.paste(Image.alpha_composite(region, tint).convert("RGB"), (x, y))
            color = highlights.get(key)
            if color is not None:
                draw.rectangle(
                    [x, y, x + thumb_px - 1, y + thumb_px - 1],
                    outline=color,
                    width=color_border_px,
                )
                draw.rectangle(
                    [x + color_border_px, y + color_border_px,
                     x + thumb_px - color_border_px - 1, y + thumb_px - color_border_px - 1],
                    outline="white",
                    width=inner_border_px,
                )
    return canvas if found else None


In [ ]:
# --- Parameters ----------------------------------------------------------- #
import json

TARGET_FOLDER = Path(
    "/data/home/mirick/ChordEdit/ablation_outputs/grid_metrics_sdturbo_top_20260626_104807"
)
THUMB_PX = 250  # per-cell size in the composite (source cells are 512x512)
GAP = 3         # light grey border (px) between cells
FIG_DPI = 200   # display resolution; figsize is derived from composite size
OVERLAY_ALPHA = 0.45  # viridis tint strength on each cell

ORIGIN_CELL = (0.0, 0.0)
# When True: greyscale ORIGIN_CELL and never give it the best-cell orange outline.
EXCLUDE_ORIGIN_CELL = True

# (t_start, t_end) -> reference border colors.
HIGHLIGHTS = {
    ORIGIN_CELL: "#000000",
    (0.9, 0.3): "#1f77b4",
}
BEST_HIGHLIGHT = "#ff7f0e"
HIGHLIGHT_COLOR_BORDER_PX = 10  # colored highlight frame thickness
HIGHLIGHT_INNER_BORDER_PX = 4    # white inner ring thickness

METRICS_CSV = next(TARGET_FOLDER.glob("id_to_metrics*.csv"))
metrics_df = pd.read_csv(METRICS_CSV)
if "whole_psnr" in metrics_df.columns:
    metrics_df = metrics_df.rename(columns={"whole_psnr": "psnr"})
metrics_df = metrics_df[metrics_df["t_delta"] == TARGET_T_DELTA]
metrics_df[TARGET_METRIC_COL] = TARGET_METRIC_COL_FN(metrics_df)

# PIE-Bench mapping: sample_id -> {original_prompt, editing_prompt, ...}
MAPPING_PATH = Path("/data/home/mirick/datasets/PIE-Bench_v1/mapping_file.json")
MAPPING = json.loads(MAPPING_PATH.read_text())


def strip_brackets(text: str) -> str:
    return text.replace("[", "").replace("]", "").strip()


>**Important**: After running the below cell, you *must* clear all outputs *before* closing the file or commiting changes to the repo. A very large number of images are processed and compiled to create these plots, and the notebook file size balloons massively after it is run.

In [ ]:
"""
Iterate over each sample and t_delta condition under TARGET_FOLDER, build a composite
grid with metric overlays and highlight borders, and display a labeled figure with source
and target prompts plus a viridis colorbar for the active metric.
"""

values = list(GRID_VALUES)
n = len(values)
centers = [GAP + i * (THUMB_PX + GAP) + THUMB_PX / 2 for i in range(n)]

origin_greyscale = {ORIGIN_CELL} if EXCLUDE_ORIGIN_CELL else set()
best_exclude = ORIGIN_CELL if EXCLUDE_ORIGIN_CELL else None

sample_dirs = sorted(p for p in TARGET_FOLDER.iterdir() if p.is_dir() and p.name != "plots")

# NOTE: Only plot the first sample for testing.
for sample_dir in sample_dirs:
    for condition_dir in sorted(
        p for p in sample_dir.iterdir() if p.is_dir() and p.name.startswith("t_delta_")
    ):
        t_delta = float(condition_dir.name.replace("t_delta_", "").replace("p", "."))
        if t_delta != TARGET_T_DELTA:
            continue
        parts = sample_dir.name.split("_")
        sample_id = parts[-1]

        scores = grid_scores(metrics_df, sample_id, t_delta, TARGET_METRIC_COL)
        highlights = {
            **HIGHLIGHTS,
            **best_highlight(scores, color=BEST_HIGHLIGHT, exclude=best_exclude),
        }
        composite = build_grid_image(
            condition_dir / "cells",
            values,
            THUMB_PX,
            GAP,
            highlights=highlights,
            color_border_px=HIGHLIGHT_COLOR_BORDER_PX,
            inner_border_px=HIGHLIGHT_INNER_BORDER_PX,
            t_delta=t_delta,
            scores=scores,
            overlay_alpha=OVERLAY_ALPHA,
            greyscale_keys=origin_greyscale,
        )
        if composite is None:
            continue

        # Folder name: <prefix>_<name...>_<count>_<sample_id>.
        # e.g. "1_change_object_80_111000000000" -> id=111000000000, cat=1_change_object
        category = "_".join(parts[:-2])
        meta = MAPPING.get(sample_id, {})
        source_prompt = strip_brackets(meta.get("original_prompt", ""))
        target_prompt = strip_brackets(meta.get("editing_prompt", ""))
        title = (
            f'Images for {sample_id} from {category}\n'
            f'Source Prompt: "{source_prompt}"\n'
            f'Target Prompt: "{target_prompt}"\n'
            f'{TARGET_METRIC_COL_LABEL}'
        )

        fig_w, fig_h = composite.width / FIG_DPI, composite.height / FIG_DPI
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=FIG_DPI)
        ax.imshow(composite, interpolation="none")
        ax.set_xticks(centers, [f"{v:.1f}" for v in values])
        ax.set_yticks(centers, [f"{v:.1f}" for v in reversed(values)])
        ax.set_xlabel(f"t_start, $\\delta={float(t_delta):.2f}$")
        ax.set_ylabel(f"t_end, $\\delta={float(t_delta):.2f}$")
        ax.set_title(title, fontsize=10)

        
        if scores:
            score_vals = list(scores.values())
            sm = ScalarMappable(
                cmap="viridis",
                norm=Normalize(vmin=min(score_vals), vmax=max(score_vals)),
            )
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(TARGET_METRIC_COL)
